### Chains Basics - Practice & Exercises

In this notebook, you will practice building basic LCEL chains and sequences.

#### Part 1: Follow-Along Practice

##### Method 1: LCEL Basics

##### Goal / Question
Connect a prompt template, chat model, and string output parser using LangChain Expression Language (LCEL) to generate joke text.

##### Sample Input
```python
{"topic": "lawyers", "joke_count": 3}
```

##### Sample Output
A plain text string containing 3 jokes about lawyers.

##### Plan / Steps
1. Load environment variables.
2. Initialize the model (`ChatOpenAI`) and prompt template.
3. Combine prompt, model, and `StrOutputParser` using the LCEL pipe operator (`|`).
4. Invoke the chain passing input values in a dictionary.

In [ ]:
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()
model = ChatOpenAI(model="gpt-4o")

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian who tells jokes about {topic}."),
    ("human", "Tell me {joke_count} jokes."),
])

chain = prompt_template | model | StrOutputParser()
result = chain.invoke({"topic": "lawyers", "joke_count": 3})
print(result)

##### Method 2: Under the Hood (RunnableSequence)

##### Goal / Question
Peel back the LCEL syntax by manually creating execution blocks using `RunnableLambda` and linking them using `RunnableSequence`.

##### Sample Input
```python
{"topic": "lawyers", "joke_count": 3}
```

##### Sample Output
A plain text string containing 3 jokes about lawyers.

##### Plan / Steps
1. Define individual runnables for formatting the prompt, invoking the model, and parsing the output using `RunnableLambda`.
2. Group the runnables sequentially using `RunnableSequence(first=..., middle=[...], last=...)`.
3. Invoke the manually constructed sequence.

In [ ]:
from langchain.schema.runnable import RunnableLambda, RunnableSequence

format_prompt = RunnableLambda(lambda x: prompt_template.format_prompt(**x))
invoke_model = RunnableLambda(lambda x: model.invoke(x.to_messages()))
parse_output = RunnableLambda(lambda x: x.content)

chain_seq = RunnableSequence(first=format_prompt, middle=[invoke_model], last=parse_output)
response = chain_seq.invoke({"topic": "lawyers", "joke_count": 3})
print(response)

#### Part 2: Practice Exercises

##### Exercise: Translate Joke Chain

##### Goal / Question
Build a sequential LCEL chain that generates a joke about a topic, and then pipes that joke into a translation step to output the joke in Spanish.

##### Sample Input
```python
{"topic": "electricity"}
```

##### Sample Output
A short science joke about electricity translated into Spanish.

##### Plan / Steps
1. Define a prompt `joke_prompt` asking for a science joke about `{topic}`.
2. Define a translation prompt `translation_prompt` asking to translate `{joke}` into Spanish.
3. Create a sub-chain `joke_chain` to generate and parse the joke string.
4. Combine the sub-chain with the translation prompt using a dictionary to map inputs: `full_chain = {"joke": joke_chain} | translation_prompt | model | StrOutputParser()`.
5. Invoke the chain passing a topic.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
joke_prompt = ChatPromptTemplate.from_template("Tell me a short science joke about {topic}.")
translation_prompt = ChatPromptTemplate.from_template("Translate this joke into Spanish: {joke}")

joke_chain = joke_prompt | model | StrOutputParser()

full_chain = {"joke": joke_chain} | translation_prompt | model | StrOutputParser()
print(full_chain.invoke({"topic": "electricity"}))
```
</details>